In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np

In [2]:
behavior = pd.read_csv(
    r"D:\NudgeIQ\data\processed\behavioral_intelligence.csv"
)

print(behavior.shape)
behavior.head()

(11162, 37)


,age,job,marital,education,default,balance,housing,loan,contact,day,...,Financial_Stability,Contact_Intensity,Relationship_History,Investment_Experience,Responsiveness,Investment_Readiness,Financial_Index,Engagement_Index,Investor_Profile_Index,Behavioral_Intelligence_Score
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,...,0.069866,0.719228,-0.36326,1.029603,1.930226,1.282201,0.069866,0.425828,1.282201,0.832651
1,56,admin.,married,secondary,no,45,no,no,unknown,5,...,-0.124253,0.719228,-0.36326,0.814669,3.154612,1.107073,-0.124253,0.425828,1.107073,0.652918
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,...,-0.234045,0.719228,-0.36326,-0.260003,2.929901,0.231430,-0.234045,0.425828,0.231430,0.203709
3,55,services,married,secondary,no,2476,yes,no,unknown,5,...,0.107537,0.719228,-0.36326,0.743024,0.596366,1.048697,0.107537,0.425828,1.048697,0.747817
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,...,-0.084883,0.995583,-0.36326,1.524871,0.867171,1.856072,-0.084883,0.672507,1.856072,1.137478


In [3]:
behavior["Financial_Level"] = pd.cut(
    behavior["Financial_Index"],
    bins=[-np.inf, -0.5, 0.5, np.inf],
    labels=["Low", "Medium", "High"]
)

behavior["Engagement_Level"] = pd.cut(
    behavior["Engagement_Index"],
    bins=[-np.inf, -0.5, 0.5, np.inf],
    labels=["Low", "Medium", "High"]
)

behavior["Profile_Level"] = pd.cut(
    behavior["Investor_Profile_Index"],
    bins=[-np.inf, -0.5, 0.5, np.inf],
    labels=["Low", "Medium", "High"]
)

In [4]:
behavior[
    [
        "Financial_Level",
        "Engagement_Level",
        "Profile_Level"
    ]
].head()

,Financial_Level,Engagement_Level,Profile_Level
0,Medium,Medium,High
1,Medium,Medium,High
2,Medium,Medium,Medium
3,Medium,Medium,High
4,Medium,High,High


In [5]:
def assign_persona(financial, engagement, profile):

    if financial == "High" and engagement == "High" and profile == "High":
        return "Premium Investor"

    elif financial == "High" and engagement in ["Medium", "High"] and profile in ["Medium", "High"]:
        return "Growth Investor"

    elif financial == "Medium" and engagement == "High" and profile in ["Medium", "High"]:
        return "Balanced Investor"

    elif financial == "Medium" and engagement == "Medium" and profile == "High":
        return "Balanced Investor"

    elif financial == "Medium" and engagement == "Medium" and profile == "Medium":
        return "General Investor"

    elif financial == "Low" and engagement == "High":
        return "Potential Investor"

    elif financial == "High" and engagement == "Low":
        return "Dormant Wealth Holder"

    elif engagement == "Low":
        return "Low Engagement User"

    else:
        return "Emerging Investor"

In [6]:
behavior["Persona"] = behavior.apply(
    lambda x: assign_persona(
        x["Financial_Level"],
        x["Engagement_Level"],
        x["Profile_Level"]
    ),
    axis=1
)

In [7]:
print(behavior["Persona"].value_counts())

Persona
Low Engagement User      3488
Emerging Investor        2211
Balanced Investor        1882
General Investor         1180
Potential Investor        834
Growth Investor           676
Dormant Wealth Holder     668
Premium Investor          223
Name: count, dtype: int64


In [8]:
persona_matrix = (
    behavior.groupby(
        ["Financial_Level", "Engagement_Level", "Profile_Level"]
    )
    .size()
    .reset_index(name="Customers")
    .sort_values("Customers", ascending=False)
)

persona_matrix

,Financial_Level,Engagement_Level,Profile_Level,Customers
13,Medium,Medium,Medium,1180
9,Medium,Low,Low,1168
10,Medium,Low,Medium,948
17,Medium,High,High,724
14,Medium,Medium,High,697
12,Medium,Medium,Low,693
0,Low,Low,Low,634
4,Low,Medium,Medium,573
16,Medium,High,Medium,461
3,Low,Medium,Low,405


In [10]:
persona_rules = {
    ("High","High","High"): "Premium Investor",
    ("High","High","Medium"): "Growth Investor",
    ("High","High","Low"): "Growth Investor",

    ("High","Medium","High"): "Growth Investor",
    ("High","Medium","Medium"): "Balanced Investor",
    ("High","Medium","Low"): "Dormant Wealth Holder",

    ("High","Low","High"): "Dormant Wealth Holder",
    ("High","Low","Medium"): "Dormant Wealth Holder",
    ("High","Low","Low"): "Dormant Wealth Holder",

    ("Medium","High","High"): "Balanced Investor",
    ("Medium","High","Medium"): "Balanced Investor",
    ("Medium","High","Low"): "Emerging Investor",

    ("Medium","Medium","High"): "Balanced Investor",
    ("Medium","Medium","Medium"): "General Investor",
    ("Medium","Medium","Low"): "General Investor",

    ("Medium","Low","High"): "Potential Investor",
    ("Medium","Low","Medium"): "Low Engagement User",
    ("Medium","Low","Low"): "Low Engagement User",

    ("Low","High","High"): "Potential Investor",
    ("Low","High","Medium"): "Emerging Investor",
    ("Low","High","Low"): "Emerging Investor",

    ("Low","Medium","High"): "Potential Investor",
    ("Low","Medium","Medium"): "Emerging Investor",
    ("Low","Medium","Low"): "Low Engagement User",

    ("Low","Low","High"): "Potential Investor",
    ("Low","Low","Medium"): "Low Engagement User",
    ("Low","Low","Low"): "Low Engagement User",
}

In [11]:
behavior["Persona"] = behavior.apply(
    lambda row: persona_rules[
        (
            row["Financial_Level"],
            row["Engagement_Level"],
            row["Profile_Level"],
        )
    ],
    axis=1,
)

In [12]:
print(behavior["Persona"].value_counts())

Persona
Low Engagement User      3505
Balanced Investor        2151
General Investor         1873
Emerging Investor        1168
Potential Investor        992
Dormant Wealth Holder     817
Growth Investor           433
Premium Investor          223
Name: count, dtype: int64


In [5]:
import pandas as pd
from pathlib import Path

# Project root = D:\NudgeIQ
project_root = Path.cwd().parent

behavior_path = project_root / "data" / "processed" / "behavioral_intelligence.csv"
rules_path = project_root / "data" / "processed" / "persona_rules.csv"

behavior = pd.read_csv(behavior_path)
rules = pd.read_csv(rules_path)

print("Behavior:", behavior.shape)
print("Rules:", rules.shape)
print("\nRule columns:")
print(rules.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: 'd:\\NudgeIQ\\data\\processed\\persona_rules.csv'

In [6]:
from pathlib import Path

project_root = Path.cwd().parent
processed = project_root / "data" / "processed"

print("Processed folder:")
print(processed)

print("\nFiles actually visible to Python:")
for f in processed.iterdir():
    print(repr(f.name))

Processed folder:
d:\NudgeIQ\data\processed

Files actually visible to Python:
'behavioral_data.csv'
'behavioral_features.csv'
'behavioral_intelligence.csv'
'persona_rules .csv'
'rule_engine_output.csv'


In [7]:
print("\nCSV files:")
for f in processed.glob("*.csv"):
    print(repr(f.name))


CSV files:
'behavioral_data.csv'
'behavioral_features.csv'
'behavioral_intelligence.csv'
'persona_rules .csv'
'rule_engine_output.csv'


In [8]:
rules_path = project_root / "data" / "processed" / "persona_rules.csv"

rules = pd.read_csv(rules_path)

print("Rules:", rules.shape)
print(rules.columns.tolist())

Rules: (27, 7)
['Financial_Level', 'Engagement_Level', 'Profile_Level', 'Persona', 'Tier', 'Priority', 'Action']


In [10]:
import numpy as np

behavior["Financial_Level"] = pd.cut(
    behavior["Financial_Index"],
    bins=[-np.inf, -0.5, 0.5, np.inf],
    labels=["Low", "Medium", "High"]
)

behavior["Engagement_Level"] = pd.cut(
    behavior["Engagement_Index"],
    bins=[-np.inf, -0.5, 0.5, np.inf],
    labels=["Low", "Medium", "High"]
)

behavior["Profile_Level"] = pd.cut(
    behavior["Investor_Profile_Index"],
    bins=[-np.inf, -0.5, 0.5, np.inf],
    labels=["Low", "Medium", "High"]
)

print(
    behavior[
        ["Financial_Level", "Engagement_Level", "Profile_Level"]
    ].head()
)

  Financial_Level Engagement_Level Profile_Level
0          Medium           Medium          High
1          Medium           Medium          High
2          Medium           Medium        Medium
3          Medium           Medium          High
4          Medium             High          High


In [11]:
print("Financial:")
print(behavior["Financial_Level"].value_counts())

print("\nEngagement:")
print(behavior["Engagement_Level"].value_counts())

print("\nProfile:")
print(behavior["Profile_Level"].value_counts())

Financial:
Financial_Level
Medium    6340
Low       3080
High      1742
Name: count, dtype: int64

Engagement:
Engagement_Level
Medium    4467
Low       4156
High      2539
Name: count, dtype: int64

Profile:
Profile_Level
Medium    4511
Low       3589
High      3062
Name: count, dtype: int64


In [12]:
behavior = behavior.merge(
    rules,
    on=[
        "Financial_Level",
        "Engagement_Level",
        "Profile_Level"
    ],
    how="left"
)

print("Rows:", len(behavior))
print("Unmatched:", behavior["Persona"].isna().sum())
print("Unique Personas:", behavior["Persona"].nunique())

Rows: 11162
Unmatched: 0
Unique Personas: 9


In [13]:
behavior["Rule_Match_Status"] = behavior["Persona"].notna().map(
    {
        True: "Matched",
        False: "Unmatched"
    }
)

print(behavior["Rule_Match_Status"].value_counts())

Rule_Match_Status
Matched    11162
Name: count, dtype: int64


In [14]:
print(
    behavior[
        [
            "Financial_Level",
            "Engagement_Level",
            "Profile_Level",
            "Persona",
            "Tier",
            "Priority",
            "Action",
            "Rule_Match_Status"
        ]
    ].head(10).to_string(index=False)
)

Financial_Level Engagement_Level Profile_Level            Persona   Tier Priority                                    Action Rule_Match_Status
         Medium           Medium          High  Balanced Investor Tier 2   Medium    Cross-sell balanced portfolio products           Matched
         Medium           Medium          High  Balanced Investor Tier 2   Medium    Cross-sell balanced portfolio products           Matched
         Medium           Medium        Medium   General Investor Tier 2   Medium  Nurture with periodic offers and reviews           Matched
         Medium           Medium          High  Balanced Investor Tier 2   Medium    Cross-sell balanced portfolio products           Matched
         Medium             High          High  Balanced Investor Tier 2   Medium    Cross-sell balanced portfolio products           Matched
            Low             High          High Potential Investor Tier 3   Medium              Targeted conversion campaign           Matched
      

In [15]:
print("=== RULE ENGINE VALIDATION ===")

print("Total customers:", len(behavior))
print("Unmatched:", behavior["Persona"].isna().sum())
print("Unique personas:", behavior["Persona"].nunique())
print("Matched:", (behavior["Rule_Match_Status"] == "Matched").sum())

print("\n=== PERSONA DISTRIBUTION ===")
print(behavior["Persona"].value_counts())

print("\n=== TIER DISTRIBUTION ===")
print(behavior["Tier"].value_counts())

print("\n=== PRIORITY DISTRIBUTION ===")
print(behavior["Priority"].value_counts())

=== RULE ENGINE VALIDATION ===
Total customers: 11162
Unmatched: 0
Unique personas: 9
Matched: 11162

=== PERSONA DISTRIBUTION ===
Persona
Inactive Investor        2378
Balanced Investor        2151
General Investor         1873
Low Engagement User      1455
Emerging Investor        1168
Potential Investor        926
Dormant Wealth Holder     555
Growth Investor           433
Premium Investor          223
Name: count, dtype: int64

=== TIER DISTRIBUTION ===
Tier
Tier 4    4388
Tier 2    4024
Tier 3    2094
Tier 1     656
Name: count, dtype: int64

=== PRIORITY DISTRIBUTION ===
Priority
Medium    8496
Low       1455
High      1211
Name: count, dtype: int64


In [16]:
output_path = project_root / "data" / "processed" / "rule_engine_output.csv"

behavior.to_csv(output_path, index=False)

print("✅ FINAL RULE ENGINE OUTPUT SAVED")
print(output_path)

✅ FINAL RULE ENGINE OUTPUT SAVED
d:\NudgeIQ\data\processed\rule_engine_output.csv


In [17]:
corr = behavior[
    [
        "Financial_Index",
        "Engagement_Index",
        "Investor_Profile_Index"
    ]
].corr()

print(corr.to_string())

                        Financial_Index  Engagement_Index  Investor_Profile_Index
Financial_Index                1.000000         -0.038193                0.107399
Engagement_Index              -0.038193          1.000000                0.481668
Investor_Profile_Index         0.107399          0.481668                1.000000


In [18]:
print(
    "Financial ↔ Investor Profile:",
    round(
        behavior["Financial_Index"].corr(
            behavior["Investor_Profile_Index"]
        ),
        3
    )
)

Financial ↔ Investor Profile: 0.107
